## Ekman Theory

### **Wind Speed at Staggered Grid Points**

The wind speed magnitude at u-points (east-west faces) and v-points (north-south faces) is computed by interpolating the perpendicular component and applying:

$$
\text{ws}_u = \sqrt{u^2 + v_{\text{on\_u}}^2}, \quad \text{ws}_v = \sqrt{u_{\text{on\_v}}^2 + v^2}
$$

where $v_{\text{on\_u}}$ is $v$ averaged to u-points in the $\xi$ (longitude) direction, and $u_{\text{on\_v}}$ is $u$ averaged to v-points in the $\eta$ (latitude) direction.

### **Surface Wind Stress**

Wind stress components at the staggered grid faces are computed using the bulk formula:

$$
\tau_x^{(u)} = \rho_{\text{air}}\, C_d\, \text{ws}_u \cdot u, \quad \tau_y^{(v)} = \rho_{\text{air}}\, C_d\, \text{ws}_v \cdot v
$$

where:
- $\rho_{\text{air}} = 1.22\,\text{kg}\,\text{m}^{-3}$ is air density,
- $C_d = 1.3 \times 10^{-3}$ is the drag coefficient.

These are then averaged to the rho-points (cell centers):

$$
\tau_x^{(\rho)} = \frac{1}{2}\left(\tau_x^{(u)}_{i} + \tau_x^{(u)}_{i+1}\right), \qquad \tau_y^{(\rho)} = \frac{1}{2}\left(\tau_y^{(v)}_{j} + \tau_y^{(v)}_{j+1}\right)
$$

### **Grid Metrics (Δx, Δy)**

Horizontal grid spacings on the spherical Earth (radius $A = 6{,}371{,}000\,\text{m}$) are:

$$
\Delta x = A\,\cos(\phi)\,\Delta\lambda, \quad \Delta y = A\,\Delta\phi
$$

where $\phi$ is latitude (radians), $\lambda$ is longitude (radians), and $\Delta\lambda$, $\Delta\phi$ are finite differences along $\xi$ and $\eta$ directions.

### **Curl of Wind Stress**

The vertical component of the curl (vorticity of the wind stress) is computed via centered finite differences:

$$
\nabla \times \boldsymbol{\tau} = \frac{\partial \tau_y}{\partial x} - \frac{\partial \tau_x}{\partial y}
$$

Discretized on the interior cell centers:

$$
\bigl(\nabla \times \boldsymbol{\tau}\bigr)_{i,j} = \frac{\tau_y^{(\rho)}_{i+1,j} - \tau_y^{(\rho)}_{i,j}}{\Delta x_{i,j}} - \frac{\tau_x^{(\rho)}_{i,j+1} - \tau_x^{(\rho)}_{i,j}}{\Delta y_{i,j}}
$$

### **Coriolis Parameter**

The Coriolis parameter at cell centers is:

$$
f = 2\,\Omega\,\sin(\phi)
$$

where $\Omega = 7.2921 \times 10^{-5}\,\text{rad}\,\text{s}^{-1}$ is Earth's angular velocity, and $\phi$ is the latitude averaged over the four surrounding rho-points. Values where $|f| < 10^{-10}$ (near the equator) are set to NaN to avoid division by zero.

### **Ekman Pumping Velocity**

The vertical Ekman pumping velocity (positive upward) at the base of the Ekman layer is:

$$
w_{\text{Ek}} = \frac{1}{\rho_w\,f}\,\bigl(\nabla \times \boldsymbol{\tau}\bigr)_z
$$

where $\rho_w = 1025\,\text{kg}\,\text{m}^{-3}$ is seawater density.

### **Cell Area**

The area of each interior cell (for spatial integrals) is:

$$
A_{\text{cell}} = \Delta x \cdot \Delta y
$$

computed at the curl centers by averaging the edge spacings from neighboring cells.

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import re
from xml.etree import ElementTree as ET
from matplotlib.path import Path
import pandas as pd
from scipy.io import loadmat
from scipy.interpolate import RegularGridInterpolator

In [2]:
from dask.distributed import Client
from dask import array as da
client = Client(n_workers=4, threads_per_worker=3, memory_limit=20e9)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 74.51 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37761,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 12
Started: Just now,Total memory: 74.51 GiB
Comm: tcp://127.0.0.1:35833,Total threads: 3
Dashboard: http://127.0.0.1:35911/status,Memory: 18.63 GiB
Nanny: tcp://127.0.0.1:38951,


### Functions

In [3]:
#### Inshore mask
## kml to struct
import re
from xml.etree import ElementTree as ET
import numpy as np

def kml2struct(kml_file):
    """
    Import a .kml file as a list of dictionary structures with fields:
    Geometry, Name, Description, Lon, Lat, and BoundingBox.
    """
    # Read the KML file
    try:
        with open(kml_file, 'r', encoding='utf-8') as file:
            txt = file.read()
    except Exception as e:
        raise FileNotFoundError(f"Unable to open file {kml_file}: {e}")

    # Regular expression to match Placemark tags
    expr = r"<Placemark.+?>.+?</Placemark>"
    object_strings = re.findall(expr, txt, re.DOTALL)
    
    kml_struct = []

    for obj_str in object_strings:
        # Extract Name
        name_match = re.search(r"<name.*?>(.*?)</name>", obj_str, re.DOTALL)
        name = name_match.group(1).strip() if name_match else "undefined"

        # Extract Description
        desc_match = re.search(r"<description.*?>(.*?)</description>", obj_str, re.DOTALL)
        desc = desc_match.group(1).strip() if desc_match else ""

        # Determine Geometry Type
        geometry = ""
        if "<Point" in obj_str:
            geometry = "Point"
        elif "<LineString" in obj_str:
            geometry = "Line"
        elif "<Polygon" in obj_str:
            geometry = "Polygon"

        # Extract Coordinates
        coord_match = re.search(r"<coordinates.*?>(.*?)</coordinates>", obj_str, re.DOTALL)
        if not coord_match:
            continue  # Skip if no coordinates are found
        coord_str = coord_match.group(1).strip()
        coord_list = np.array([list(map(float, coord.split(','))) for coord in coord_str.split()])

        # Separate Lon, Lat, and handle Polygons
        lon = coord_list[:, 0]
        lat = coord_list[:, 1]
        if geometry == "Polygon":
            # Close the polygon by appending NaN
            lon = np.append(lon, np.nan)
            lat = np.append(lat, np.nan)

        # Create BoundingBox
        bounding_box = [[lon.min(), lat.min()], [lon.max(), lat.max()]]

        # Append to kml_struct
        kml_struct.append({
            "Geometry": geometry,
            "Name": name,
            "Description": desc,
            "Lon": lon,
            "Lat": lat,
            "BoundingBox": bounding_box
        })

    return kml_struct

In [4]:
arch_kml_zona1 = "/gxfs_work/geomar/smomw662/NHCS/hindcast/CROCO_BioEBUS_1990-2010/indices/BK111km.kml"
R1 = kml2struct(arch_kml_zona1)

# Extract Lon and Lat from the first polygon
lonb1 = R1[0]["Lon"]
latb1 = R1[0]["Lat"]

In [5]:
ds_ = xr.open_dataset('/gxfs_work/geomar/smomw662/NHCS/hincast_1980-2015/croco_avg_Y1980M01.nc', 
                     chunks = {'time':1})
ds_

<xarray.Dataset> Size: 891MB
Dimensions:     (xi_rho: 602, xi_u: 601, eta_rho: 542, eta_v: 541, s_rho: 32,
                 s_w: 33, time: 1, auxil: 4)
Coordinates: (12/13)
  * xi_rho      (xi_rho) float64 5kB 1.0 2.0 3.0 4.0 ... 599.0 600.0 601.0 602.0
  * xi_u        (xi_u) float64 5kB 1.5 2.5 3.5 4.5 ... 598.5 599.5 600.5 601.5
  * eta_rho     (eta_rho) float64 4kB 1.0 2.0 3.0 4.0 ... 540.0 541.0 542.0
  * eta_v       (eta_v) float64 4kB 1.5 2.5 3.5 4.5 ... 538.5 539.5 540.5 541.5
  * s_rho       (s_rho) float64 256B -0.9844 -0.9531 ... -0.04688 -0.01562
  * s_w         (s_w) float64 264B -1.0 -0.9688 -0.9375 ... -0.0625 -0.03125 0.0
    ...          ...
    lat_rho     (eta_rho, xi_rho) float64 3MB dask.array<chunksize=(542, 602), meta=np.ndarray>
    lon_u       (eta_rho, xi_u) float64 3MB dask.array<chunksize=(542, 601), meta=np.ndarray>
    lat_u       (eta_rho, xi_u) float64 3MB dask.array<chunksize=(542, 601), meta=np.ndarray>
    lon_v       (eta_v, xi_rho) float64 3MB dask.array<chunksize=(541, 602), meta=np.ndarray>
    lat_v       (eta_v, xi_rho) float64 3MB dask.array<chunksize=(541, 602), meta=np.ndarray>
  * time        (time) float32 4B 1.339e+06
Dimensions without coordinates: auxil
Data variables: (12/54)
    spherical   |S1 1B ...
    xl          float64 8B ...
    el          float64 8B ...
    Vtransform  float64 8B ...
    sc_r        (s_rho) float64 256B dask.array<chunksize=(32,), meta=np.ndarray>
    sc_w        (s_w) float64 264B dask.array<chunksize=(33,), meta=np.ndarray>
    ...          ...
    swflux      (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
    radsw       (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
    shflx_rlw   (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
    shflx_lat   (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
    shflx_sen   (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
    hel         (time, eta_rho, xi_rho) float32 1MB dask.array<chunksize=(1, 542, 602), meta=np.ndarray>
Attributes: (12/57)
    type:           ROMS restart file
    title:          Peru UW Interannual Run
    date:           
    rst_file:       croco_rst.nc
    his_file:       croco_his.nc
    avg_file:       croco_avg.nc
    ...             ...
    gamma2_expl:    Slipperiness parameter
    x_sponge:       0.0
    v_sponge:       0.0
    sponge_expl:    Sponge parameters : extent (m) & viscosity (m2.s-1)
    SRCS:           main.F step.F read_inp.F timers_roms.F init_scalars.F ini...
    CPP-options:    REGIONAL PERU_UW MPI OBC_WEST OBC_NORTH OBC_SOUTH BIOLOGY...

In [6]:
LON, LAT = np.meshgrid(ds_.lon_rho.compute().values[0,:],ds_.lat_rho.compute().values[:,0])
# Create a path from the polygon coordinates
polygon = Path(np.column_stack((lonb1, latb1)))

# Flatten the LON and LAT to create coordinate pairs
lonlat_points = np.column_stack((LON.ravel(), LAT.ravel()))

# Check which points are inside the polygon
mask = polygon.contains_points(lonlat_points).reshape(LON.shape)

# Convert the mask to NaN for the outshore region
inshore_mask = np.where(mask, 1, np.nan)

mask_rho = ds_.mask_rho.values 
mask_nan = np.where(mask_rho, 1, np.nan)

## Data
* For only 1 netcdf file

In [7]:
import glob

files = sorted(glob.glob('/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y*M*_uwnd_vwnd.nc'))
print(len(files), files[:5], files[-5:])

# Peek at first two
for f in files[:3]:
    with xr.open_dataset(f) as ds:
        print(f, ds['bulk_time'].load().values[:3], ds['bulk_time'].load().values[-3:])

432 ['/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M10_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M11_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M12_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M1_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M2_uwnd_vwnd.nc'] ['/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M5_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M6_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M7_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M8_uwnd_vwnd.nc', '/gxfs_work/geomar/

In [8]:
from tqdm import tqdm
for f in tqdm(files):
    fn = f[-21:-12]
    # print(f)
    # print(fn)

100%|█████████████████████████████████████████████████████████████████████████████| 432/432 [00:00<00:00, 2570126.71it/s]


In [10]:
from tqdm import tqdm

for f in tqdm(files):
    fn = f[-21:-12]
    ## Open the data 
    ds = xr.open_dataset(f, chunks=-1)
    ds = ds[['uwnd', 'vwnd']]
    ds = ds.load()
    
    origin = np.datetime64('1980-01-01T00:00:00')  # example; choose your true origin
    abs_time = origin + ds.bulk_time
    
    #################
    #### Panda dates
    pdt = pd.DatetimeIndex(abs_time.values.astype('datetime64[ns]'))
    year  = xr.DataArray(pdt.year,  coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='year')
    month = xr.DataArray(pdt.month, coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='month')
    day   = xr.DataArray(pdt.day,   coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='day')
    hour   = xr.DataArray(pdt.hour,   coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='hour')
    
    ##################
    ## Assign coords
    ds = ds.assign_coords(
        year = (('bulk_time',), year.values),
        month = (('bulk_time',), month.values),
        day = (('bulk_time',), day.values),
        hour = (('bulk_time',), hour.values),
    )
    
    time = xr.DataArray(abs_time.astype('datetime64[ns]'),
                        coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='time')
    
    # Add it as a coordinate and keep bulk_time as-is
    ds = ds.assign_coords(time=('bulk_time', time.values))
    
    #########################
    ### Pump calculation####
    # A: Earth radius (meters)
    A = 6371000.0
    
    # Ensure uwnd and vwnd are in the expected order and dims (time, eta_u, xi_u), (time, eta_v, xi_v)
    u = xr.DataArray(
        ds['uwnd'].data,
        dims=('time','eta_u','xi_u'),
        coords=dict(
            time=ds.time.data,
            lat_u=(('eta_u','xi_u'), ds_.lat_u.compute().values),
            lon_u=(('eta_u','xi_u'), ds_.lon_u.compute().values),
        ),
        name='u10'
    )
    
    v = xr.DataArray(
        ds['vwnd'].data,
        dims=('time','eta_v','xi_v'),
        coords=dict(
            time=ds.time.data,
            lat_v=(('eta_v','xi_v'), ds_.lat_v.compute().values),
            lon_v=(('eta_v','xi_v'), ds_.lon_v.compute().values),
        ),
        name='v10'
    )
    
    # ----------------------------
    # Compute wind speed at faces with proper alignment
    # ----------------------------
    # Map v to u points: average in xi (x) direction -> loses 1 cell in xi
    v_on_u = 0.5 * (
        v.isel(xi_v=slice(0, -1)).rename({'eta_v':'eta_u', 'xi_v':'xi_u'}) +
        v.isel(xi_v=slice(1,  None)).rename({'eta_v':'eta_u', 'xi_v':'xi_u'})
    )
    
    # Align shapes by trimming to common interior
    eta_u_size = min(u.sizes['eta_u'], v_on_u.sizes['eta_u'])
    xi_u_size  = min(u.sizes['xi_u'],  v_on_u.sizes['xi_u'])
    
    u_c      = u.isel(eta_u=slice(0, eta_u_size), xi_u=slice(0, xi_u_size))
    v_on_u_c = v_on_u.isel(eta_u=slice(0, eta_u_size), xi_u=slice(0, xi_u_size))
    
    # Map u to v points: average in eta (y) direction -> loses 1 cell in eta
    u_on_v = 0.5 * (
        u.isel(eta_u=slice(0, -1)).rename({'eta_u':'eta_v', 'xi_u':'xi_v'}) +
        u.isel(eta_u=slice(1,  None)).rename({'eta_u':'eta_v', 'xi_u':'xi_v'})
    )
    
    # Align shapes
    eta_v_size = min(v.sizes['eta_v'], u_on_v.sizes['eta_v'])
    xi_v_size  = min(v.sizes['xi_v'],  u_on_v.sizes['xi_v'])
    
    v_c      = v.isel(eta_v=slice(0, eta_v_size), xi_v=slice(0, xi_v_size))
    u_on_v_c = u_on_v.isel(eta_v=slice(0, eta_v_size), xi_v=slice(0, xi_v_size))
    
    # Wind speed magnitudes at u and v faces
    ws_u = xr.apply_ufunc(np.hypot, u_c, v_on_u_c,dask='allowed')  # (time, eta_u_size, xi_u_size)
    ws_v = xr.apply_ufunc(np.hypot, u_on_v_c, v_c,dask='allowed')  # (time, eta_v_size, xi_v_size)
    
    # ----------------------------
    # Surface wind stresses at faces
    # ----------------------------
    rho_air = 1.22
    Cd = 1.3e-3
    taux_u = rho_air * Cd * ws_u * u_c
    tauy_v = rho_air * Cd * ws_v * v_c
    
    # ----------------------------
    # Average stresses to rho points
    # ----------------------------
    # tau_x from u -> rho (average in xi)
    taux_rho = 0.5 * (
        taux_u.isel(xi_u=slice(0, -1)).rename({'eta_u':'eta_rho','xi_u':'xi_rho'}) +
        taux_u.isel(xi_u=slice(1,  None)).rename({'eta_u':'eta_rho','xi_u':'xi_rho'})
    )
    
    # tau_y from v -> rho (average in eta)
    tauy_rho = 0.5 * (
        tauy_v.isel(eta_v=slice(0, -1)).rename({'eta_v':'eta_rho','xi_v':'xi_rho'}) +
        tauy_v.isel(eta_v=slice(1,  None)).rename({'eta_v':'eta_rho','xi_v':'xi_rho'})
    )
    
    # Align tau arrays to common rho grid size
    eta_rho_size = min(taux_rho.sizes['eta_rho'], tauy_rho.sizes['eta_rho'])
    xi_rho_size  = min(taux_rho.sizes['xi_rho'],  tauy_rho.sizes['xi_rho'])
    
    taux_rho = taux_rho.isel(eta_rho=slice(0, eta_rho_size), xi_rho=slice(0, xi_rho_size))
    tauy_rho = tauy_rho.isel(eta_rho=slice(0, eta_rho_size), xi_rho=slice(0, xi_rho_size))
    
    # Attach 2D rho coords (trimmed to match)
    lat_rho = xr.DataArray(
        ds_.lat_rho.compute().values[:eta_rho_size, :xi_rho_size], 
        dims=('eta_rho','xi_rho'), 
        name='lat'
    )
    lon_rho = xr.DataArray(
        ds_.lon_rho.compute().values[:eta_rho_size, :xi_rho_size], 
        dims=('eta_rho','xi_rho'), 
        name='lon'
    )
    taux_rho = taux_rho.assign_coords(lat=lat_rho, lon=lon_rho)
    tauy_rho = tauy_rho.assign_coords(lat=lat_rho, lon=lon_rho)
    
    # ----------------------------
    # Grid metrics (Δx, Δy) on rho grid
    # ----------------------------
    # Differences along xi (longitude) and eta (latitude)
    dlam = np.deg2rad(lon_rho.diff('xi_rho'))                   # (eta_rho, xi_rho-1)
    lat_x = 0.5 * (lat_rho.isel(xi_rho=slice(0,-1)) + lat_rho.isel(xi_rho=slice(1,None)))
    dx = A * np.cos(np.deg2rad(lat_x)) * dlam                   # (eta_rho, xi_rho-1)
    
    dphi = np.deg2rad(lat_rho.diff('eta_rho'))                  # (eta_rho-1, xi_rho)
    dy = A * dphi                                               # (eta_rho-1, xi_rho)

    # ----------------------------
    # Centered derivatives to common centers
    # ----------------------------
    # d(tau_y)/dx lives on (eta_rho, xi_rho-1)
    dtauy_dx_edge = (tauy_rho.isel(xi_rho=slice(1, None)) - tauy_rho.isel(xi_rho=slice(0, -1))) / dx
    
    # d(tau_x)/dy lives on (eta_rho-1, xi_rho)
    dtaux_dy_edge = (taux_rho.isel(eta_rho=slice(1, None)) - taux_rho.isel(eta_rho=slice(0, -1))) / dy
    
    # Average both to interior cell centers (eta_rho-1, xi_rho-1)
    dtauy_dx_c = 0.5 * (dtauy_dx_edge.isel(eta_rho=slice(0, -1)) + dtauy_dx_edge.isel(eta_rho=slice(1, None)))
    dtaux_dy_c = 0.5 * (dtaux_dy_edge.isel(xi_rho=slice(0, -1)) + dtaux_dy_edge.isel(xi_rho=slice(1, None)))
    
    # Ensure same sizes for curl computation
    eta_c = min(dtauy_dx_c.sizes['eta_rho'], dtaux_dy_c.sizes['eta_rho'])
    xi_c  = min(dtauy_dx_c.sizes['xi_rho'],  dtaux_dy_c.sizes['xi_rho'])
    dtauy_dx_c = dtauy_dx_c.isel(eta_rho=slice(0, eta_c), xi_rho=slice(0, xi_c))
    dtaux_dy_c = dtaux_dy_c.isel(eta_rho=slice(0, eta_c), xi_rho=slice(0, xi_c))
    
    curl_tau = (dtauy_dx_c - dtaux_dy_c)  # (time, eta_c, xi_c)
    curl_tau.name = 'curl_tau'
    
    # ----------------------------
    # Coriolis parameter at curl centers
    # ----------------------------
    # Center lat for f (average of the four surrounding rho points)
    lat_c = 0.25 * (
        lat_rho.isel(eta_rho=slice(0, eta_c), xi_rho=slice(0, xi_c)) +
        lat_rho.isel(eta_rho=slice(1, eta_c+1), xi_rho=slice(0, xi_c)) +
        lat_rho.isel(eta_rho=slice(0, eta_c), xi_rho=slice(1, xi_c+1)) +
        lat_rho.isel(eta_rho=slice(1, eta_c+1), xi_rho=slice(1, xi_c+1))
    )
    
    Omega = 7.2921e-5
    f = 2.0 * Omega * np.sin(np.deg2rad(lat_c))
    # Avoid divide by zero near equator
    f = xr.where(np.abs(f) < 1e-10, np.nan, f)
    
    # ----------------------------
    # Ekman pumping velocity
    # ----------------------------
    rho_w = 1025.0
    w_ek = curl_tau / (rho_w * f)
    w_ek = w_ek.transpose('time','eta_rho','xi_rho')
    w_ek.name = 'w_ek'
    
    ##############################
    #### Add latitudes ##########
    
    lat_c_da = lat_c  # at curl centers
    band = (lat_c_da >= -16) & (lat_c_da <= -5)	
    
    wek_ms = w_ek * inshore_mask[:eta_c, :xi_c]* mask_nan[:eta_c, :xi_c]
    wek_m2_s = wek_ms * 111e3  # units: (m^2/s)
    
    
    #ms
    wek_ms_ = (wek_ms.where(band)).mean(dim=('xi_rho')).sortby('time')  # m2/s
    #m2/s
    wek_m2_s_ = (wek_m2_s.where(band)).mean(dim=('xi_rho')).sortby('time')  # m^2/s
    wek_m2_s_
    
    lat=lat_c_da.isel(xi_rho=0)
    wek_m2_s_['lat'] = lat
    
    wek_m2_s_.attrs["units"] = "m2/s"
    wek_m2_s_.attrs["long_name"] = "Ekman pumping (area flux)"
    
    
    wek_m2_s_ = wek_m2_s_.swap_dims({'eta_rho':'lat'})
    
    #####################
    ###### Export
    ####################
    Pump = wek_m2_s_
    Pump.drop_encoding().to_netcdf(f'Pump/Pump{fn}.nc')


100%|████████████████████████████████████████████████████████████████████████████████| 432/432 [2:26:23<00:00, 20.33s/it]
